# 🚀 Hindi NER: mBERT Fine-Tuning (Google Colab GPU Runner)

**Instructions:**
1. Go to **Runtime > Change runtime type > Select T4 GPU**.
2. Click **Runtime > Run all**.
3. When finished (~12-18 mins), it will automatically download `mbert_test_metrics.json` and `mbert_predictions.json` to your Mac.
4. Move those two downloaded files into your local `ner_using_transformers/results/` and `ner_using_transformers/results/predictions/` folders!

In [10]:
# Step 1: Install required libraries in Colab
!pip install -q transformers datasets accelerate setuptools
!pip install -q --no-deps seqeval

import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ WARNING: GPU not detected! Go to Runtime > Change runtime type > Select T4 GPU for fast training.")

  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.
CUDA Available: True
Using GPU: Tesla T4


In [11]:
# Step 2: Load the HiNER dataset directly via JSON files
from datasets import load_dataset

base_url = "https://huggingface.co/datasets/cfilt/HiNER-collapsed/resolve/main/data"
data_files = {
    "train": f"{base_url}/train.json",
    "validation": f"{base_url}/validation.json",
    "test": f"{base_url}/test.json",
}

print("Loading HiNER dataset...")
dataset = load_dataset("json", data_files=data_files)
print("✓ Dataset loaded:")
for s in dataset:
    print(f"  {s}: {len(dataset[s]):,} examples")

Loading HiNER dataset...
✓ Dataset loaded:
  train: 75,827 examples
  validation: 10,851 examples
  test: 21,657 examples


In [12]:
# Step 3: Setup Tokenizer, Subword Alignment, and Robust Metrics
from transformers import AutoTokenizer, AutoModelForTokenClassification, DataCollatorForTokenClassification, TrainingArguments, Trainer
import numpy as np
import json

LABEL_LIST = ["O", "B-PER", "I-PER", "B-LOC", "I-LOC", "B-ORG", "I-ORG"]
LABEL_TO_ID = {label: idx for idx, label in enumerate(LABEL_LIST)}
ID_TO_LABEL = {idx: label for idx, label in enumerate(LABEL_LIST)}

# Strict entity evaluation helper (works whether seqeval is installed or not)
def extract_bio_entities(labels):
    entities = []
    curr_type = None
    start = None
    for i, label in enumerate(labels):
        if label.startswith('B-'):
            if curr_type is not None:
                entities.append((curr_type, start, i - 1))
            curr_type = label[2:]
            start = i
        elif label.startswith('I-'):
            if curr_type is None or label[2:] != curr_type:
                if curr_type is not None:
                    entities.append((curr_type, start, i - 1))
                curr_type = label[2:]
                start = i
        else:
            if curr_type is not None:
                entities.append((curr_type, start, i - 1))
                curr_type = None
                start = None
    if curr_type is not None:
        entities.append((curr_type, start, len(labels) - 1))
    return set(entities)

def compute_entity_metrics(true_seqs, pred_seqs):
    try:
        from seqeval.metrics import precision_score, recall_score, f1_score, classification_report
        p = precision_score(true_seqs, pred_seqs)
        r = recall_score(true_seqs, pred_seqs)
        f = f1_score(true_seqs, pred_seqs)
        rep_dict = classification_report(true_seqs, pred_seqs, output_dict=True)
        rep_str = classification_report(true_seqs, pred_seqs)
        return p, r, f, rep_dict, rep_str
    except Exception:
        # Pure Python fallback with exact same IOB2 metric math
        tp, fp, fn = 0, 0, 0
        by_type = {}
        for t_seq, p_seq in zip(true_seqs, pred_seqs):
            t_set = extract_bio_entities(t_seq)
            p_set = extract_bio_entities(p_seq)
            tp += len(t_set & p_set)
            fp += len(p_set - t_set)
            fn += len(t_set - p_set)
            for ent_type, s, e in t_set:
                by_type.setdefault(ent_type, {'tp': 0, 'fp': 0, 'fn': 0, 'support': 0})
                by_type[ent_type]['support'] += 1
                if (ent_type, s, e) in p_set:
                    by_type[ent_type]['tp'] += 1
                else:
                    by_type[ent_type]['fn'] += 1
            for ent_type, s, e in p_set:
                by_type.setdefault(ent_type, {'tp': 0, 'fp': 0, 'fn': 0, 'support': 0})
                if (ent_type, s, e) not in t_set:
                    by_type[ent_type]['fp'] += 1
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
        rep_dict = {}
        lines = [f"{'':15s}{'precision':>10s}{'recall':>10s}{'f1-score':>10s}{'support':>10s}"]
        for ent, counts in sorted(by_type.items()):
            ep = counts['tp'] / (counts['tp'] + counts['fp']) if (counts['tp'] + counts['fp']) > 0 else 0.0
            er = counts['tp'] / (counts['tp'] + counts['fn']) if (counts['tp'] + counts['fn']) > 0 else 0.0
            ef = 2 * ep * er / (ep + er) if (ep + er) > 0 else 0.0
            rep_dict[ent] = {'precision': ep, 'recall': er, 'f1-score': ef, 'support': counts['support']}
            lines.append(f"{ent:15s}{ep:10.2f}{er:10.2f}{ef:10.2f}{counts['support']:10d}")
        rep_dict['micro avg'] = {'precision': prec, 'recall': rec, 'f1-score': f1, 'support': tp + fn}
        lines.append(f"\n{'micro avg':15s}{prec:10.2f}{rec:10.2f}{f1:10.2f}{tp+fn:10d}")
        return prec, rec, f1, rep_dict, '\n'.join(lines)

model_name = "bert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_and_align_labels(examples, max_length=128):
    tokenized = tokenizer(
        examples["tokens"],
        truncation=True,
        max_length=max_length,
        is_split_into_words=True,
    )
    all_labels = []
    for i, labels in enumerate(examples["ner_tags"]):
        word_ids = tokenized.word_ids(batch_index=i)
        label_ids = []
        prev_word_idx = None
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != prev_word_idx:
                label_ids.append(labels[word_idx])
            else:
                label_ids.append(-100)
            prev_word_idx = word_idx
        all_labels.append(label_ids)
    tokenized["labels"] = all_labels
    return tokenized

print("Tokenizing dataset with label alignment...")
tokenized_train = dataset["train"].map(tokenize_and_align_labels, batched=True)
tokenized_val = dataset["validation"].map(tokenize_and_align_labels, batched=True)
tokenized_test = dataset["test"].map(tokenize_and_align_labels, batched=True)
print("✓ Tokenization complete.")

Tokenizing dataset with label alignment...
✓ Tokenization complete.


In [13]:
# Step 4: Model Initialization & Metrics Callback
model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=len(LABEL_LIST),
    id2label=ID_TO_LABEL,
    label2id=LABEL_TO_ID,
)

def compute_metrics(eval_preds):
    predictions, labels = eval_preds
    predictions = np.argmax(predictions, axis=2)
    true_labels = []
    pred_labels = []
    for p_seq, l_seq in zip(predictions, labels):
        true_s = []
        pred_s = []
        for p, l in zip(p_seq, l_seq):
            if l != -100:
                true_s.append(ID_TO_LABEL[l])
                pred_s.append(ID_TO_LABEL[p])
        true_labels.append(true_s)
        pred_labels.append(pred_s)
    p, r, f, _, _ = compute_entity_metrics(true_labels, pred_labels)
    return {"precision": p, "recall": r, "f1": f}
print("✓ Model and metrics initialized.")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params

✓ Model and metrics initialized.


In [14]:
# Step 5: Fine-tune mBERT (3 Epochs with T4 GPU fp16 mixed-precision)
# Construct training arguments safely across different transformers versions
base_args = {
    "output_dir": "./mbert_checkpoints",
    "learning_rate": 2e-5,
    "per_device_train_batch_size": 16,
    "per_device_eval_batch_size": 32,
    "num_train_epochs": 3,
    "weight_decay": 0.01,
    "load_best_model_at_end": True,
    "metric_for_best_model": "f1",
    "greater_is_better": True,
    "fp16": torch.cuda.is_available(),
    "logging_steps": 100,
    "seed": 42,
    "report_to": "none",
}

try:
    training_args = TrainingArguments(eval_strategy="epoch", save_strategy="epoch", **base_args)
except TypeError:
    training_args = TrainingArguments(evaluation_strategy="epoch", save_strategy="epoch", **base_args)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=DataCollatorForTokenClassification(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)

print("Starting training on GPU...")
trainer.train()

Starting training on GPU...


Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,0.061864,0.061537,0.884480,0.888502,0.886487
2,0.049251,0.057481,0.893992,0.900442,0.897205
3,0.039983,0.058374,0.892944,0.906260,0.899552


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

TrainOutput(global_step=14220, training_loss=0.05518709535588695, metrics={'train_runtime': 1693.571, 'train_samples_per_second': 134.32, 'train_steps_per_second': 8.396, 'total_flos': 1.0633565849398734e+16, 'train_loss': 0.05518709535588695, 'epoch': 3.0})

In [15]:
# Step 6: Evaluate on Test Set & Generate Metrics Report
print("Running inference on Test split (21,657 sentences)...")
raw_preds = trainer.predict(tokenized_test)
test_preds_ids = np.argmax(raw_preds.predictions, axis=2)

test_true_labels = []
test_pred_labels = []
for p_seq, l_seq in zip(test_preds_ids, raw_preds.label_ids):
    t_s = []
    p_s = []
    for p, l in zip(p_seq, l_seq):
        if l != -100:
            t_s.append(ID_TO_LABEL[l])
            p_s.append(ID_TO_LABEL[p])
    test_true_labels.append(t_s)
    test_pred_labels.append(p_s)

p, r, f, rep_dict, rep_str = compute_entity_metrics(test_true_labels, test_pred_labels)

print("\n" + "=" * 60)
print("mBERT TEST EVALUATION REPORT (Entity-Level seqeval)")
print("=" * 60)
print(f"Overall Precision: {p:.4f}")
print(f"Overall Recall:    {r:.4f}")
print(f"Overall F1 Score:  {f:.4f}")
print("\nDetailed Breakdown:")
print(rep_str)

# Save metrics file
test_metrics = {
    "overall_precision": p,
    "overall_recall": r,
    "overall_f1": f,
    "report": rep_dict,
    "report_str": rep_str,
}
with open("mbert_test_metrics.json", "w", encoding="utf-8") as out_f:
    json.dump(test_metrics, out_f, indent=2)

# Save predictions file
preds_list = []
for i, ex in enumerate(dataset["test"]):
    preds_list.append({
        "tokens": ex["tokens"],
        "true_labels": test_true_labels[i],
        "pred_labels": test_pred_labels[i],
    })
with open("mbert_predictions.json", "w", encoding="utf-8") as out_f:
    json.dump(preds_list, out_f, ensure_ascii=False, indent=2)

print("\n✓ Saved mbert_test_metrics.json and mbert_predictions.json")

Running inference on Test split (21,657 sentences)...



mBERT TEST EVALUATION REPORT (Entity-Level seqeval)
Overall Precision: 0.8905
Overall Recall:    0.9060
Overall F1 Score:  0.8982

Detailed Breakdown:
                precision    recall  f1-score   support
LOC                  0.84      0.86      0.85      7421
ORG                  0.90      0.91      0.91     58706
PER                  0.87      0.89      0.88     12691

micro avg            0.89      0.91      0.90     78818

✓ Saved mbert_test_metrics.json and mbert_predictions.json


In [17]:
# Step 7: Download the resulting files to your Mac
try:
    from google.colab import files
    print("Downloading output files to your Mac...")
    files.download("mbert_test_metrics.json")
    files.download("mbert_predictions.json")
    print("✓ Triggered browser download!")
except Exception as e:
    print("Files are saved locally on the instance as 'mbert_test_metrics.json' and 'mbert_predictions.json'.")
    print(f"Manual download note: {e}")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✓ Triggered browser download!


In [18]:
mv ~/Downloads/mbert_test_metrics.json results/

mv: cannot stat '/root/Downloads/mbert_test_metrics.json': No such file or directory
